# G3 — Hub compatibility suite: do these N models share a space?

A general test. Register any number of encoders' cached vectors over the
same items, and this notebook answers three questions with measured
numbers and a stated verdict:

1. **Do they relate pairwise at all?** A direct ridge fit A→B into B's
   raw space, per ordered pair. This is context, not the verdict
   baseline — the hub's target is a whitened consensus of all N spaces,
   which is an easier thing to hit, so ratios against it can exceed
   100% and mean little. The verdict instead compares the shared hub
   against a **bespoke hub built for that pair alone**, which is the
   question that matters: does one common space cost anything?
2. **Can ONE shared space serve all of them?** Build a hub, map each
   encoder in, and ask whether two encoders' hub coordinates for the
   SAME item match — for every ordered pair, with no designated target
   space and no head to train.
3. **Is the answer real?** A random map in the same slot must fall to
   chance, or the test proves nothing.

**The core metric is target-free.** Encode item *i* with model A and
item *j* with model B; both land in the hub; ask whether the true match
(*i* = *j*) ranks first among all candidates. That works for any number
of encoders, any mix of modalities, and needs no labels beyond the item
correspondence the caches already carry.

**Pre-registered gates** (stated before you run it):

| verdict | condition |
|---|---|
| SHARED | mutual agreement ≥ 70% of a hub built for that pair alone, control at chance |
| PARTIAL | agreement clearly above chance but below the gate |
| NOT SHARED | agreement within noise of chance, or control not at chance |

A PARTIAL is a real result — it usually means the hub is carrying coarse
correspondence without pair-level resolution, which the per-pair table
will localise.

In [ ]:
import os
from pathlib import Path
STORAGE   = "drive"
DRIVE_DIR = "/content/drive/MyDrive/convergence_experiment"
LOCAL_DIR = "./convergence_data"
try:
    import google.colab                 # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
if STORAGE == "env":
    assert os.environ.get("DATA_DIR"), "STORAGE='env' but DATA_DIR unset"
elif STORAGE == "drive" and IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    os.environ["DATA_DIR"] = DRIVE_DIR
else:
    if STORAGE == "drive":
        print("not on Colab - using LOCAL_DIR")
    os.environ["DATA_DIR"] = str(Path(LOCAL_DIR).resolve())
DATA_DIR = Path(os.environ["DATA_DIR"])
print("DATA_DIR:", DATA_DIR)

## 1 · Register the models

Edit `SOURCES` only. Each entry names a cached `.npz`, the array inside
it, and a label. Row *i* of every array must be the SAME item — the
alignment check below verifies what it can and refuses to guess.

In [ ]:
import numpy as np

# name -> (filename, key inside the npz, modality tag for the report)
SOURCES = {
    "img_small": ("e1_img_ckpt_dinov2-small_cls+patch.npz", "img", "image"),
    "img_base":  ("e1_img_ckpt_dinov2-base_cls+patch.npz",  "img", "image"),
    "img_large": ("e1_img_ckpt_dinov2-large_cls+patch.npz", "img", "image"),
    "txt_bge":   ("crossmodal_pairs.npz",                   "txt", "text"),
    "txt_gpt2":  ("crossmodal_pairs_gpt2.npz",              "txt", "text"),
    # written by E1.3 - four training objectives in one hub. Both are
    # skipped automatically if E1.3 has not been run.
    "txt_bert":  ("e13_txt_bert.npz",                       "txt", "text"),
    "txt_sbert": ("e13_txt_sbert.npz",                      "txt", "text"),
}

SPACES, MODALITY, skipped = {}, {}, []
for name, (fn, key, mod) in SOURCES.items():
    f = DATA_DIR / fn
    if not f.exists():
        skipped.append(f"{name} (missing {fn})"); continue
    d = np.load(str(f))
    if key not in d.files:
        skipped.append(f"{name} (no '{key}' in {fn})"); continue
    SPACES[name] = d[key].astype(np.float64)
    MODALITY[name] = mod
assert len(SPACES) >= 2, f"need >= 2 spaces, got {list(SPACES)}"
if skipped:
    print("skipped:", "; ".join(skipped))

# --- alignment: verify what can be verified, never assume the rest ---
N = min(len(v) for v in SPACES.values())
SPACES = {k: v[:N] for k, v in SPACES.items()}
proved = []
for a in SPACES:
    for b in SPACES:
        if a >= b: continue
        if SPACES[a].shape[1] == SPACES[b].shape[1] and \
           np.allclose(SPACES[a][:N], SPACES[b][:N], atol=1e-4):
            proved.append(f"{a} == {b}")
print(f"\n{len(SPACES)} spaces on {N} common rows")
for k, v in SPACES.items():
    print(f"   {k:12s} {str(v.shape):>16s}  {MODALITY[k]}")
if proved:
    print("   identical arrays detected:", ", ".join(proved))
print("\nNOTE: row alignment across DIFFERENT encoders cannot be proved "
      "from the\nvectors alone - it is guaranteed by how the caches were "
      "built (same\ndeterministic id list, nested prefixes). If that is "
      "not true of your\nfiles, every number below is meaningless.")

rng = np.random.default_rng(0)
perm = rng.permutation(N)
N_EVAL = min(1000, N // 4)
te, tr = perm[:N_EVAL], perm[N_EVAL:]
print(f"\n{len(tr)} train / {len(te)} eval (gallery {N_EVAL}, "
      f"chance R@1 = {1/N_EVAL:.4f})")

## 2 · Per-space geometry — the shape of each encoder's output

In [ ]:
from scipy.stats import skew

def l2n(V): return V / (np.linalg.norm(V, axis=-1, keepdims=True) + 1e-12)
def ridge(X, Y, a):
    return np.linalg.solve(X.T @ X + a * np.eye(X.shape[1]), X.T @ Y)
def r_at_1(Q, G):
    return float(((l2n(Q) @ l2n(G).T).argmax(1) ==
                  np.arange(len(Q))).mean())
def effrank(V):
    Vc = V - V.mean(0)
    s = np.linalg.svd(Vc, full_matrices=False, compute_uv=False)
    p = s ** 2 / (s ** 2).sum(); p = p[p > 0]
    return float(np.exp(-(p * np.log(p)).sum()))
def paircos(V):
    Vn = l2n(V); r = np.random.default_rng(1)
    i, j = r.integers(0, len(V), 3000), r.integers(0, len(V), 3000)
    m = i != j
    return float((Vn[i[m]] * Vn[j[m]]).sum(1).mean())
def hubness(V, k=10):
    S = l2n(V) @ l2n(V).T
    np.fill_diagonal(S, -9)
    nn = np.argsort(-S, 1)[:, :k]
    return float(skew(np.bincount(nn.ravel(), minlength=len(V))))

print(f"{'space':12s} {'width':>6s} {'rows/dim':>9s} {'eff.rank':>9s} "
      f"{'% of width':>11s} {'pair-cos':>9s} {'N10 skew':>9s}")
GEOM = {}
for k, v in SPACES.items():
    er = effrank(v[tr]); w = v.shape[1]
    GEOM[k] = dict(er=er, pc=paircos(v[tr]), hub=hubness(v[te]),
                   rows_per_dim=len(tr)/w)
    print(f"{k:12s} {w:6d} {GEOM[k]['rows_per_dim']:9.1f} {er:9.1f} "
          f"{100*er/w:10.1f}% {GEOM[k]['pc']:+9.3f} {GEOM[k]['hub']:9.2f}")
print("\nlow eff.rank + high pair-cos = collapsed (cosine will struggle);")
print("high N10 skew = hubbed (a few items dominate everyone's neighbours)")

## Shape agreement — the check that needs no map

Everything below this point compares spaces *through* a fitted map. This
cell does not: each space is compared only to ITSELF (how far apart are
items i and j inside it), and the two answers are correlated across
spaces. Nothing is fitted, and the measure is invariant to orientation,
scale and dimension.

It is worth running FIRST, because it predicts what a hub can achieve.
A pair whose shapes already agree has a correspondence waiting for
coordinates; a pair at zero here has no linear structure for any hub to
find. This is also the exact quantity Gromov-Wasserstein exploits to
recover a correspondence with no pairs at all (report C.12), and the
reason whitening HURTS there — equalising variance flattens the shape
this cell measures.

In [ ]:
from scipy.stats import spearmanr

# this cell runs BEFORE the pairwise-baseline cell, so it defines the
# names list itself rather than relying on one defined later
names = list(SPACES)

SHAPE_N = min(400, len(te))          # pairwise distances are O(n^2)
_si = rng.permutation(len(te))[:SHAPE_N]
_iu = np.triu_indices(SHAPE_N, 1)

def _pdist(Z):
    Zn = l2n(Z)
    return (1.0 - Zn @ Zn.T)[_iu]

def _knn_overlap(X, Y, k=10):
    def nn(Z):
        Zn = l2n(Z); S = Zn @ Zn.T
        np.fill_diagonal(S, -9)
        return np.argsort(-S, 1)[:, :k]
    a, b = nn(X), nn(Y)
    return float(np.mean([len(set(a[i]) & set(b[i])) / k
                          for i in range(len(X))]))

_D = {k: _pdist(SPACES[k][te][_si]) for k in names}
SHAPE_RHO, SHAPE_KNN = {}, {}
print(f"shape agreement on {SHAPE_N} held-out items, NO fitted map")
print(f"(10-NN overlap by chance = {100*10/SHAPE_N:.1f}%)\n")
print(f"{'pair':26s} {'dist rho':>9s} {'10-NN overlap':>14s}"
      f"  {'reads as':>12s}")
for a in names:
    for b in names:
        if a >= b: continue
        rho = float(spearmanr(_D[a], _D[b]).statistic)
        ov = _knn_overlap(SPACES[a][te][_si], SPACES[b][te][_si])
        SHAPE_RHO[(a, b)] = rho; SHAPE_KNN[(a, b)] = ov
        tag = ("strong" if rho > 0.5 else
               "moderate" if rho > 0.25 else
               "weak" if rho > 0.1 else "none")
        print(f"{a+' <-> '+b:26s} {rho:+9.3f} {100*ov:13.1f}% "
              f"{tag:>12s}")
print("\nrho near 0 with overlap near chance = no shared shape; no hub")
print("can manufacture one. High values here are a correspondence that")
print("exists and is merely waiting for coordinates.")

In [ ]:
import matplotlib.pyplot as plt

def plot_shape_agreement(pair=None):
    """Scatter the pairwise distances of two spaces against each other,
    plus the neighbourhood overlap. No map is involved in either."""
    if pair is None:                      # default: the strongest pair
        pair = max(SHAPE_RHO, key=SHAPE_RHO.get)
    a, b = pair
    rho, ov = SHAPE_RHO[pair], SHAPE_KNN[pair]
    fig, ax = plt.subplots(1, 3, figsize=(13, 3.8))

    s = rng.permutation(len(_D[a]))[:6000]
    ax[0].scatter(_D[a][s], _D[b][s], s=3, alpha=0.20, c="#1a5276",
                  edgecolors="none")
    z = np.polyfit(_D[a][s], _D[b][s], 1)
    xs = np.linspace(_D[a].min(), _D[a].max(), 50)
    ax[0].plot(xs, np.polyval(z, xs), c="#c0392b", lw=1.6, ls="--")
    ax[0].set_xlabel(f"cosine distance in {a}", fontsize=8.4)
    ax[0].set_ylabel(f"cosine distance in {b}", fontsize=8.4)
    ax[0].set_title(f"same pairs, two spaces\nSpearman rho = {rho:+.3f}",
                    fontsize=9.6, color="#1a1a2e")
    ax[0].grid(alpha=0.2); ax[0].tick_params(labelsize=7.4)

    ax[1].bar(["shared\n10-NN", "chance"], [100*ov, 100*10/SHAPE_N],
              color=["#0f766e", "#95a5a6"], width=0.55)
    for xi, v in enumerate([100*ov, 100*10/SHAPE_N]):
        ax[1].text(xi, v + 1.2, f"{v:.1f}%", ha="center", fontsize=10.5,
                   color="#1a1a2e", weight="bold")
    ax[1].set_ylabel("% of 10 nearest neighbours\nshared across spaces",
                     fontsize=8.2)
    ax[1].set_title("neighbourhoods agree", fontsize=9.6, color="#1a1a2e")
    ax[1].grid(alpha=0.2, axis="y"); ax[1].tick_params(labelsize=8)

    ps = sorted(SHAPE_RHO.items(), key=lambda kv: -kv[1])
    lbl = [f"{x[0][0][:9]}\n{x[0][1][:9]}" for x in ps]
    ax[2].bar(range(len(ps)), [x[1] for x in ps], color="#1a5276",
              width=0.6)
    ax[2].set_xticks(range(len(ps)))
    ax[2].set_xticklabels(lbl, fontsize=6.4)
    ax[2].axhline(0, c="#95a5a6", lw=1)
    ax[2].set_ylabel("distance-rank correlation", fontsize=8.2)
    ax[2].set_title("every pair, ranked", fontsize=9.6, color="#1a1a2e")
    ax[2].grid(alpha=0.2, axis="y"); ax[2].tick_params(labelsize=7.4)

    for x in ax:
        x.spines["top"].set_visible(False); x.spines["right"].set_visible(False)
    fig.suptitle(f"Shape agreement WITHOUT any fitted map — {a} vs {b}",
                 fontsize=11, color="#1a1a2e")
    fig.subplots_adjust(left=0.08, right=0.97, top=0.82, bottom=0.20,
                        wspace=0.30)
    plt.show()

plot_shape_agreement()

## 3 · Baseline: do they relate pairwise at all?

Before asking about a shared space, establish that each ordered pair
relates directly. This is the number every hub result is scored against
— a hub cannot be expected to beat a map fitted for one specific pair.

In [ ]:
ALPHA = 1e-2
names = list(SPACES)
DIRECT = {}
print("direct A -> B, fitted per pair (R@1 into B's own space)\n")
print(f"{'':12s}" + "".join(f"{c:>11s}" for c in names))
for a in names:
    row = f"{a:12s}"
    for b in names:
        if a == b:
            row += f"{'-':>11s}"; continue
        W = ridge(SPACES[a][tr], SPACES[b][tr], ALPHA)
        v = r_at_1(SPACES[a][te] @ W, SPACES[b][te])
        DIRECT[(a, b)] = v
        row += f"{v:11.3f}"
    print(row)
print(f"\nchance = {1/N_EVAL:.4f}. Pairs at chance here cannot be helped "
      f"by any hub -\nthey have no linear correspondence to share.")

## 4 · Build the hub, and sweep its width

The hub is a whitened PCA of all spaces concatenated after per-space
standardisation, so no encoder dominates by width or scale. Width is
swept rather than chosen: too narrow and it cannot carry the spaces, too
wide and whitened PCA amplifies noise directions that destroy cosine
agreement. The sweep enforces at least 5 training rows per hub
dimension.

In [ ]:
_ref = np.hstack([(SPACES[k][tr] - SPACES[k][tr].mean(0)) /
                  (SPACES[k][tr].std(0).mean() + 1e-12) for k in names])
_mu = _ref.mean(0)
_U, _sv, _VT = np.linalg.svd(_ref - _mu, full_matrices=False)
print(f"concatenated reference width {_ref.shape[1]}, "
      f"{len(_sv)} components available")

def hub_maps(K, alpha=ALPHA):
    Bs = _VT[:K].T / (_sv[:K] / np.sqrt(len(_ref)))
    def co(w):
        X = np.hstack([(SPACES[k][w] - SPACES[k][tr].mean(0)) /
                       (SPACES[k][tr].std(0).mean() + 1e-12) for k in names])
        return (X - _mu) @ Bs
    htr = co(tr)
    return co, {k: ridge(SPACES[k][tr], htr, alpha) for k in names}

def mean_agreement(co, TH):
    vals = []
    for a in names:
        for b in names:
            if a == b: continue
            vals.append(r_at_1(SPACES[a][te] @ TH[a], SPACES[b][te] @ TH[b]))
    return float(np.mean(vals))

DIMS = [k for k in (128, 256, 512, 768, 1024, 1536) if k <= len(_sv)]
print(f"\n{'HUB_DIM':>8s} {'rows/dim':>9s} {'mean agreement':>15s}")
best = (-1, None)
for K in DIMS:
    if len(tr) / K < 5:
        print(f"{K:8d} {len(tr)/K:9.1f}   skipped (below 5 rows/dim)")
        continue
    co, TH = hub_maps(K)
    m = mean_agreement(co, TH)
    print(f"{K:8d} {len(tr)/K:9.1f} {m:15.3f}")
    if m > best[0]: best = (m, K)
HUB_DIM = best[1]
print(f"\nchosen HUB_DIM = {HUB_DIM} (mean agreement {best[0]:.3f})")
CO, TO_HUB = hub_maps(HUB_DIM)

## 5 · The shared-space test, per pair — plus controls and a verdict

In [ ]:
# ============================================================== #
# Two baselines, and only ONE of them is a fair verdict.
#
#  (a) RAW DIRECT (computed earlier): fit A->B and match in B's own
#      raw space. Useful context, but NOT comparable to the hub: the
#      hub target is a whitened consensus of all N spaces, which is an
#      easier thing to hit than one encoder's raw geometry. Ratios far
#      above 100% here are expected and mean little - and they are
#      LARGEST exactly where the raw target is worst (a collapsed
#      space), because the hub's whitening rescues it.
#
#  (b) BESPOKE HUB (computed below): build a hub from JUST that pair,
#      at the same width, and compare. Same metric, same kind of
#      target. This isolates the question that matters - does ONE
#      shared space cost anything against a space built for the pair?
#      That is the verdict metric.
# ============================================================== #

def bespoke_maps(a, b, K):
    ref = np.hstack([(SPACES[k][tr] - SPACES[k][tr].mean(0)) /
                     (SPACES[k][tr].std(0).mean() + 1e-12) for k in (a, b)])
    mu = ref.mean(0)
    _u, sv, VT = np.linalg.svd(ref - mu, full_matrices=False)
    Kb = min(K, len(sv))
    Bs = VT[:Kb].T / (sv[:Kb] / np.sqrt(len(ref)))
    def co(w):
        X = np.hstack([(SPACES[k][w] - SPACES[k][tr].mean(0)) /
                       (SPACES[k][tr].std(0).mean() + 1e-12) for k in (a, b)])
        return (X - mu) @ Bs
    htr = co(tr)
    return {k: ridge(SPACES[k][tr], htr, ALPHA) for k in (a, b)}

AGREE, RETAIN, BESPOKE = {}, {}, {}
print(f"mutual agreement in the SHARED hub, HUB_DIM={HUB_DIM} "
      f"(chance {1/N_EVAL:.4f})\n")
print(f"{'':12s}" + "".join(f"{c:>11s}" for c in names))
for a in names:
    row = f"{a:12s}"
    for b in names:
        v = r_at_1(SPACES[a][te] @ TO_HUB[a], SPACES[b][te] @ TO_HUB[b])
        if a != b:
            AGREE[(a, b)] = v
        row += f"{v:11.3f}"
    print(row)

print("\ncomputing bespoke per-pair hubs for the fair baseline...")
for a in names:
    for b in names:
        if a == b: continue
        TH2 = bespoke_maps(a, b, HUB_DIM)
        BESPOKE[(a, b)] = r_at_1(SPACES[a][te] @ TH2[a],
                                 SPACES[b][te] @ TH2[b])
        RETAIN[(a, b)] = AGREE[(a, b)] / max(BESPOKE[(a, b)], 1e-9)

print(f"\n{'pair':26s} {'bespoke':>9s} {'shared':>9s} {'retained':>9s}"
      f" {'raw direct':>11s}  {'verdict':>12s}")
verdicts = []
for (a, b), v in AGREE.items():
    bes, d = BESPOKE[(a, b)], DIRECT[(a, b)]
    if bes < 10 / N_EVAL:
        tag = "no baseline"
    elif RETAIN[(a, b)] >= 0.70:
        tag = "SHARED"
    elif v > 10 / N_EVAL:
        tag = "PARTIAL"
    else:
        tag = "NOT SHARED"
    verdicts.append(tag)
    # flag pairs the hub RESCUES relative to their raw-space fit
    mark = "  *" if d > 0 and v / max(d, 1e-9) > 2.0 else ""
    print(f"{a+' -> '+b:26s} {bes:9.3f} {v:9.3f} "
          f"{100*RETAIN[(a,b)]:8.1f}% {d:11.3f}  {tag:>12s}{mark}")

resc = [(a, b) for (a, b) in AGREE
        if DIRECT[(a, b)] > 0 and AGREE[(a, b)] / DIRECT[(a, b)] > 2.0]
if resc:
    print("\n* the hub more than DOUBLES the raw-space result for these "
          "pairs.")
    print("  That is not hub magic - it is the whitening inside the hub "
          "basis")
    print("  rescuing a collapsed TARGET space. Check the geometry table: "
          "the")
    print("  target of every starred pair should have low effective rank "
          "and")
    print("  high pair-cosine. Section C.11 measured the same effect "
          "explicitly.")
    for a, b in resc[:6]:
        print(f"    {a} -> {b}: raw {DIRECT[(a,b)]:.3f} -> hub "
              f"{AGREE[(a,b)]:.3f}  "
              f"({AGREE[(a,b)]/DIRECT[(a,b)]:.1f}x)")

# control: one encoder gets a random map into the hub
ctrl = []
for a in names[:1]:
    for b in names[1:]:
        R = np.random.default_rng(9).standard_normal(TO_HUB[b].shape) / \
            np.sqrt(SPACES[b].shape[1])
        ctrl.append(r_at_1(SPACES[a][te] @ TO_HUB[a], SPACES[b][te] @ R))
ctrl_max = float(np.max(ctrl))
print(f"\ncontrol (random map substituted): max R@1 {ctrl_max:.4f} "
      f"vs chance {1/N_EVAL:.4f} -> "
      + ("OK" if ctrl_max < 5 / N_EVAL else "FAILED - results not valid"))

n_shared = verdicts.count("SHARED"); n_part = verdicts.count("PARTIAL")
mean_ret = 100 * float(np.mean([RETAIN[k] for k in AGREE]))
print("\n" + "=" * 62)
if ctrl_max >= 5 / N_EVAL:
    print("OVERALL: INVALID - the control did not fall to chance")
elif n_shared == len(verdicts):
    print(f"OVERALL: SHARED SPACE - all {len(verdicts)} ordered pairs "
          f"retain >= 70%")
    print(f"of a hub built for that pair alone. Mean retention "
          f"{mean_ret:.1f}%:")
    print("one coordinate system costs essentially nothing against "
          "bespoke ones.")
elif n_shared + n_part == len(verdicts):
    print(f"OVERALL: PARTIAL - {n_shared} shared, {n_part} partial "
          f"(mean retention {mean_ret:.1f}%)")
else:
    print(f"OVERALL: MIXED - {n_shared} shared, {n_part} partial, "
          f"{verdicts.count('NOT SHARED')} not shared")
print("=" * 62)

## 6 · Dynamic demo — one input, every model's answer

Pick any held-out item. It is encoded by each registered model, each
projects into the shared hub, and every model then answers "which item
is this?" from its own side. Agreement between the columns is the shared
space being used, item by item, rather than summarised.

In [ ]:
def demo(item=None, top=3, seed=None):
    """One item in, every model's mutual answer out."""
    r = np.random.default_rng(seed)
    q = int(r.integers(len(te))) if item is None else int(item)
    print("=" * 70)
    print(f"INPUT: held-out item #{q} (row {te[q]} of the cache)\n")
    H = {k: SPACES[k][te] @ TO_HUB[k] for k in names}
    for a in names:
        qa = l2n(H[a][q:q+1])
        line = []
        for b in names:
            sc = (qa @ l2n(H[b]).T).ravel()
            order = np.argsort(-sc)[:top]
            hit = "OK " if order[0] == q else "   "
            line.append(f"{b}: {hit}#{order[0]} ({sc[order[0]]:.3f})")
        print(f"  encoded by {a:11s} -> " + " | ".join(line))
    print("\n  'OK #q' means that model recovered the SAME item the input "
          "came from,\n  using only hub coordinates - no shared weights, "
          "no joint training.")

demo(seed=7)
demo(seed=11)

## How to read the output

The per-pair table is the result; the overall verdict is a summary of
it. Three patterns recur:

- **High retention across the board** — one coordinate system genuinely
  serves every registered model.
- **High within a modality, lower across** — expected, and worth
  reporting as such rather than as a failure; cross-modal pairs start
  from a lower direct baseline too, which the retention column already
  accounts for.
- **A pair that is PARTIAL while its direct fit is strong** — the hub is
  losing that pair's specific resolution. Check that pair's entry in the
  geometry table first: a collapsed or heavily hubbed space is usually
  the one being lost.

Report the control alongside every number. Without it, an agreement
figure means nothing.